# Module 2 — Building with LLM APIs: Practical OpenAI Lab

A hands-on companion to `Module-2-Building-with-LLM-APIs-Detailed.md`. Follow the units in order, run one cell at a time, inspect outputs, and answer each **Try it** prompt. This notebook uses LangChain with OpenAI, while retaining an optional local-model comparison from Unit 2.2.

**You will build:** a conversational assistant, streaming display, validated extractor, safe tool loop, chains, retry/fallback handling, a resumable batch, request reporting, and a small support assistant.

**Before running:** API calls cost money. Start with single examples, set a budget in your OpenAI account, and check model availability and current [pricing](https://developers.openai.com/api/docs/pricing). Batch exercises and the capstone make multiple calls. Never put a key, personal data, or private customer records in notebook cells or saved outputs. Results vary; assertions against model-generated content may fail and are learning signals.

Each unit builds on the setup and earlier definitions; after restarting the kernel run setup and prerequisite cells again. `▶` denotes a paid OpenAI call. Exercises labeled optional require additional setup or multiple calls. No API calls execute until you run their cells.


## 0. Setup

Use Python 3.10+ and select your virtual environment as the notebook kernel. Run the next cell once, then restart the kernel if packages were upgraded. In a terminal you can instead run `python -m pip install -U "langchain[openai]" python-dotenv pydantic ipykernel`.

Set `OPENAI_API_KEY` in your environment, or place it in a `.env` file in this folder or a parent folder using `OPENAI_API_KEY=...`. Ensure `.env` is ignored by Git. Do not paste your key into this notebook. The model can be changed using `OPENAI_MODEL`; choose an accessible model that supports tool calling and structured outputs. See [OpenAI models](https://developers.openai.com/api/docs/models) and [LangChain ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai).


In [1]:
%pip install -U "langchain[openai]" python-dotenv pydantic ipykernel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 577.1/577.1 kB 29.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 933.7/933.7 kB 31.6 MB/s  0:00:00
  Attempting uninstall: typing-inspection
    Found existing installation: typing-inspection 0.4.2
    Uninstalling typing-inspection-0.4.2:
      Successfully uninstalled typing-inspection-0.4.2
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.2
    Uninstalling python-dotenv-1.2.2:
      Successfully uninstalled python-dotenv-1.2.2
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.18
    Uninstalling langchain-protocol-0.0.18:
      Successfully uninstalled langchain-protocol-0.0.18
  Attemp

In [2]:
import os, json, time, statistics, re
from pathlib import Path
from typing import Literal
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field, ValidationError, field_validator
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool

load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in your environment or .env, then rerun this cell.")
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-6-luna")
# Avoid temperature: support varies by model. Limit retries and set a timeout.
model = ChatOpenAI(model=MODEL_NAME, max_tokens=1024, timeout=45, max_retries=2, stream_usage=True)
print("Model:", MODEL_NAME, "| key loaded:", bool(os.getenv("OPENAI_API_KEY")))


Model: gpt-6-luna | key loaded: True


## 2.1 — Calls, messages, memory, and templates (Day 7)

`invoke` returns an AI message, not just text. Inspect usage and provider metadata; field names depend on the model/API. A truncation or incomplete status means the answer may need a higher output limit.


In [3]:
# ▶ First call
reply = model.invoke("In two short sentences, explain what an API is.")
print("Answer:", reply.text)
print("Usage:", reply.usage_metadata)
print("Metadata:", reply.response_metadata)


Answer: An API is a set of rules that lets different software programs communicate. It defines how one program can request data or services from another.
Usage: {'input_tokens': 17, 'output_tokens': 30, 'total_tokens': 47, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Metadata: {'token_usage': {'completion_tokens': 30, 'prompt_tokens': 17, 'total_tokens': 47, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EeYTpbETrDM2L0bKnPOGv3MN2BZur', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}


In [ ]:
# ▶ Explicit conversation history
history = [SystemMessage(content="You are a friendly tutor. Reply in at most two sentences.")]
def chat(user_text):
    history.append(HumanMessage(content=user_text))
    try:
        response = model.invoke(history)
    except Exception:
        history.pop()  # do not leave an unanswered turn in history
        raise
    history.append(response)
    return response
print(chat("My name is Asha and I work in logistics.").text)
print(chat("What is my name and profession?").text)
print("Message count:", len(history))


In [ ]:
# ▶ Compare: only the latest message is sent; the model has no prior context.
no_history = model.invoke([HumanMessage(content="What is my name and profession?")])
print(no_history.text)


In [ ]:
# ▶ Three reusable placeholders
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Answer in {language} using at most two sentences."),
    ("human", "{question}"),
])
for values in [
    {"role": "science tutor", "language": "English", "question": "What is a token?"},
    {"role": "career coach", "language": "Hindi", "question": "How can I learn consistently?"},
]:
    messages = prompt.invoke(values)
    print("Filled prompt:", messages.to_messages())
    print("Answer:", model.invoke(messages).text, "\n")


**Try it:** Compare the two memory examples. Change the role and language in the template. Inspect `history` and the usage from successive calls. Why can the input cost grow on every turn? Optional: consult the model page before experimenting with `reasoning_effort`; supported values and cost vary.


## 2.2 — Compare models (Day 8)

The main path uses OpenAI. An optional local Ollama model lets you explore the original module's open-weight exercise; it needs separate installation, memory, and a model license review. [Ollama library](https://ollama.com/library). The sample questions below are synthetic; replace them with ten non-sensitive questions relevant to your work. Record quality manually using a consistent rubric; latency is affected by network and warm-up.


In [ ]:
questions = [
    "Explain an API to a beginner.", "What is a rate limit?", "Write a polite delay notice.",
    "Summarize: We shipped Monday. The package arrives Friday.", "Classify as positive or negative: This was disappointing.",
    "Give two ways to protect an API key.", "Explain the difference between validation and correctness.",
    "A user asks for a refund on another person's account. What should software verify?",
    "Why can retries duplicate a payment?", "Design a simple evaluation for an order-status assistant."
]
def compare_models(models, questions):
    rows = []
    for name, candidate in models.items():
        for i, question in enumerate(questions, 1):
            started = time.perf_counter()
            try:
                result = candidate.invoke(question)
                rows.append({"model": name, "question_id": i, "seconds": round(time.perf_counter()-started, 2),
                             "answer": result.text, "usage": result.usage_metadata, "quality": "UNSCORED"})
            except Exception as exc:
                rows.append({"model": name, "question_id": i, "seconds": round(time.perf_counter()-started, 2),
                             "error_type": type(exc).__name__, "quality": "FAILED"})
    return rows
# ▶ Optional: 10 calls per model. Start with questions[:2] if preferred.
# comparison = compare_models({"openai": model}, questions)
# for row in comparison: print(row)


In [ ]:
# Optional LOCAL experiment: install Ollama, run `ollama pull llama3.2`, start Ollama,
# and run `%pip install -U langchain-ollama` in a separate cell first.
# from langchain_ollama import ChatOllama
# local = ChatOllama(model="llama3.2", temperature=0)
# comparison = compare_models({"openai": model, "local": local}, questions)
# for row in comparison: print(row)  # grade good/acceptable/bad; estimate API cost separately


**Try it:** Grade all ten answers and record local hardware cost separately from API charges. Which tasks need a stronger model? Consider accuracy, privacy, latency, license, and maintenance. Hosted open-weight endpoints need their own provider credentials and configuration.


## 2.3 — Streaming (Day 9)

Streaming improves time to first visible text; it does not guarantee a shorter total duration. Chunks can be combined to preserve the full message, including metadata when available.


In [4]:
# ▶ Stream, measure first visible text, and reconstruct the message
start = time.perf_counter()
first_text_s = None
full = None
for chunk in model.stream("Explain a library catalog in four short sentences."):
    full = chunk if full is None else full + chunk
    if chunk.text:
        if first_text_s is None: first_text_s = time.perf_counter() - start
        print(chunk.text, end="", flush=True)
elapsed = time.perf_counter() - start
print("\nFirst text:", first_text_s, "Total seconds:", round(elapsed, 2))
print("Usage:", full.usage_metadata if full is not None else None)


A library catalog is an organized list of a library’s materials. It includes books, films, and other resources. You can search by title, author, or subject. It shows where to find each item.
First text: 1.6965702919987962 Total seconds: 1.92
Usage: {'input_tokens': 15, 'output_tokens': 86, 'total_tokens': 101, 'input_token_details': {'audio': 0, 'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 35}}


In [ ]:
# ▶ Append the assembled response, then ask a dependent follow-up
if full is not None:
    stream_history = [HumanMessage(content="Explain a library catalog in four short sentences."), full,
                      HumanMessage(content="Give one example related to your previous explanation.")]
    print(model.invoke(stream_history).text)


In [ ]:
# ▶ Stream a chain and preserve partial text on interruption or failure
slogan_chain = ChatPromptTemplate.from_template("Write a two-line slogan for {product}.") | model | StrOutputParser()
def stream_safely(runnable, input_value):
    parts = []
    status = "complete"
    try:
        for piece in runnable.stream(input_value):
            parts.append(piece)
            print(piece, end="", flush=True)
    except KeyboardInterrupt:
        status = "cancelled"
    except Exception as exc:
        status = f"interrupted ({type(exc).__name__})"
    return {"text": "".join(parts), "status": status}
stream_result = stream_safely(slogan_chain, {"product": "a reusable water bottle"})
print("\nStatus:", stream_result["status"], "Characters:", len(stream_result["text"]))


In [ ]:
# ▶ Async streaming in Jupyter: use top-level await, not asyncio.run().
async def async_demo():
    async for chunk in model.astream("Give two facts about the Moon."):
        print(chunk.text, end="", flush=True)
    print()
await async_demo()


**Try it:** Compare `invoke` latency with streaming for a short and a long answer. Interrupt a stream with the notebook stop button; inspect the returned partial text if the kernel permits recovery. Decide whether your application should discard or label partial answers.


In [ ]:
# ▶ Compare separate requests; generated lengths and network conditions can differ.
def compare_stream_timing(question):
    start = time.perf_counter()
    model.invoke(question)
    invoke_s = time.perf_counter() - start
    start = time.perf_counter(); first = None
    for chunk in model.stream(question):
        if chunk.text and first is None: first = time.perf_counter() - start
    print({"invoke_total_s": round(invoke_s, 2), "stream_first_text_s": first,
           "stream_total_s": round(time.perf_counter()-start, 2)})
# compare_stream_timing("Explain an API in one sentence.")
# compare_stream_timing("Explain an API in five short paragraphs.")


## 2.4 — Structured output and evaluation (Days 10–11)

A schema checks shape and rules, not factual accuracy. This example includes a real date, a positive total, and nested optional details. Unknown values must not be invented; this particular schema rejects a missing total and routes it for review.


In [ ]:
from datetime import date
class LineItem(BaseModel):
    name: str
    quantity: int = Field(gt=0)
    price: float = Field(ge=0, description="Price of this line, not necessarily unit price")
class Receipt(BaseModel):
    vendor: str = Field(min_length=1)
    date: str = Field(description="Purchase date in ISO YYYY-MM-DD format")
    total: float = Field(gt=0, description="Total paid without currency symbol")
    currency: Literal["INR", "USD", "EUR"]
    category: Literal["food", "travel", "office", "other"]
    items: list[LineItem] = Field(description="Items if explicitly provided, otherwise empty")
    tip: float | None = Field(description="Tip if explicitly shown, otherwise null")
    @field_validator("date")
    @classmethod
    def valid_date(cls, value):
        if date.fromisoformat(value) > date.today():
            raise ValueError("Purchase date cannot be in the future")
        return value
extractor = model.with_structured_output(Receipt, include_raw=True)


In [ ]:
# ▶ Generate, inspect raw usage and handle validation/parsing failure
receipt_text = "Cafe Mocha, 2025-11-03. Total ₹640 paid by UPI. No tip mentioned."
out = extractor.invoke("Extract only facts stated in this receipt: " + receipt_text)
print("Parsed:", out["parsed"])
print("Parsing error:", out["parsing_error"])
print("Usage:", out["raw"].usage_metadata if out.get("raw") else None)


In [ ]:
# Offline: schema-valid is not necessarily correct
for total in [640, 64, -640]:
    try:
        r = Receipt(vendor="Cafe Mocha", date="2025-11-03", total=total, currency="INR",
                    category="food", items=[], tip=None)
        print(total, "valid; matches source?", r.total == 640)
    except ValidationError:
        print(total, "invalid")


In [ ]:
# ▶ Optional: 10 calls; compare independently specified ground truth, not model self-grading.
receipt_cases = [
 ("Cafe Mocha, 2025-01-02, total INR 640", "Cafe Mocha", 640, "INR"),
 ("Metro Rail, 2025-02-03, paid INR 80", "Metro Rail", 80, "INR"),
 ("Paper Shop, 2025-03-04, TOTAL USD 12.50", "Paper Shop", 12.5, "USD"),
 ("Bakery Lane; 2025-04-05; EUR 9", "Bakery Lane", 9, "EUR"),
 ("Taxi Co | 2025-05-06 | INR 350", "Taxi Co", 350, "INR"),
 ("Book Hub on 2025-06-07: USD 21", "Book Hub", 21, "USD"),
 ("Green Cafe 2025-07-08. Paid EUR 7.25", "Green Cafe", 7.25, "EUR"),
 ("Office Mart 2025-08-09; total INR 1200", "Office Mart", 1200, "INR"),
 ("Bus Co, 2025-09-10, fare USD 5", "Bus Co", 5, "USD"),
 ("Market Square, 2025-10-11, total EUR 30", "Market Square", 30, "EUR"),
]
def score_receipts(cases):
    results = []
    for text, vendor, total, currency in cases:
        try:
            out = extractor.invoke("Extract only stated facts: " + text)
            parsed = out["parsed"]
            correct = parsed is not None and (parsed.vendor, parsed.total, parsed.currency) == (vendor, total, currency)
            results.append({"valid": parsed is not None, "correct": correct, "parsed": parsed,
                            "error": str(out["parsing_error"]) if out["parsing_error"] else None})
        except Exception as exc:
            results.append({"valid": False, "correct": False, "error": type(exc).__name__})
    print("Valid:", sum(r["valid"] for r in results), "/", len(results),
          "Correct on three checked fields:", sum(r["correct"] for r in results), "/", len(results))
    return results
# receipt_results = score_receipts(receipt_cases)
# for case, result in zip(receipt_cases, receipt_results): print(case, result)


**Try it:** Add missing-total, foreign-currency, and ambiguous examples. Compare every expected field, including date and category, before relying on this metric. When extraction fails, retry with specific validation feedback only if appropriate; otherwise reject or send for human review. A schema-valid hallucinated value remains wrong.


In [ ]:
# ▶ Validation feedback retry: only parsing/validation problems trigger another attempt.
def extract_with_feedback(text, max_attempts=2):
    feedback = ""
    for attempt in range(1, max_attempts + 1):
        out = extractor.invoke("Extract only facts in this receipt. Never invent missing details.\n" + text + feedback)
        if out["parsed"] is not None:
            return out["parsed"]
        error = out.get("parsing_error")
        print("Rejected attempt", attempt, ":", type(error).__name__)
        feedback = "\nPrevious result failed schema validation. Correct the fields if supported by the receipt; otherwise leave unsupported details unknown."
    return None  # queue for review instead of inventing values
# print(extract_with_feedback(receipt_text))


## 2.5 — Tool calling (Days 12–13)

The model requests a function; your application validates and executes it. These rates and weather are **fictional static demo data**, never live facts. The refund tool simulates a request; it does not change a real account.


In [ ]:
@tool
def demo_rate(base: str, target: str) -> str:
    """Look up a fictional demo exchange rate, not a current market rate."""
    rates = {("USD", "INR"): 83.2, ("EUR", "INR"): 90.1}
    pair = (base.upper(), target.upper())
    if pair not in rates: raise ValueError("Demo pair unavailable")
    return f"Demo only: 1 {pair[0]} = {rates[pair]} {pair[1]}"
@tool
def demo_weather(city: str) -> str:
    """Look up fictional demo weather for Surat or Mumbai, never live weather."""
    data = {"surat": "34 C, humid", "mumbai": "32 C, cloudy"}
    if city.lower() not in data: raise ValueError("Demo city unavailable")
    return "Demo only: " + data[city.lower()]
@tool
def simulate_refund(order_id: str, amount: float) -> str:
    """Simulate a refund request; amounts above 5000 require human approval. No real refund is made."""
    if not re.fullmatch(r"ORD-\d+", order_id): raise ValueError("Invalid order ID")
    if not 0 < amount <= 5000: raise ValueError("Amount outside allowed range; human review required")
    return f"Simulation only: refund request for {order_id}, amount {amount}"
tool_list = [demo_rate, demo_weather, simulate_refund]
def tool_loop(bound_model, tools, messages, max_rounds=4, verbose=True):
    lookup = {t.name: t for t in tools}
    messages = list(messages)
    for _ in range(max_rounds):
        ai = bound_model.invoke(messages)
        messages.append(ai)
        if verbose: print("Assistant tool requests:", ai.tool_calls)
        if not ai.tool_calls: return ai.text, messages
        for call in ai.tool_calls:
            try:
                if call["name"] not in lookup: raise ValueError("Unknown tool")
                result = lookup[call["name"]].invoke(call)
            except Exception as exc:
                result = ToolMessage(content=f"Tool error: {type(exc).__name__}; request could not be completed.",
                                     tool_call_id=call["id"], name=call["name"])
            messages.append(result)
            if verbose: print("Tool result:", result.content)
    return "Could not finish within the tool-call limit.", messages
bound = model.bind_tools(tool_list)


In [ ]:
# ▶ Try both tools, neither tool, and an unsupported pair (up to several calls each).
for q in ["Using demo data, convert 100 USD to INR and report Surat weather.",
          "What is the capital of Japan?", "Using demo data, convert EUR to USD."]:
    print("\nQuestion:", q)
    answer, trace = tool_loop(bound, tool_list, [SystemMessage(content="Clearly label demo data. Never invent tool results."), HumanMessage(content=q)])
    print("Answer:", answer)
    print("Roles:", [m.type for m in trace])


In [ ]:
# Offline guardrail test: code enforces the limit even if a model requests it.
try:
    print(simulate_refund.invoke({"order_id": "ORD-123", "amount": 50000}))
except Exception as exc:
    print("Blocked:", type(exc).__name__, str(exc))


**Try it:** Print `trace` and match every `tool_call_id` to its request. Test different tool descriptions and multiple calls. Production write tools must also authenticate the caller, authorize the specific resource using trusted application state, and handle idempotency; the simulated refund is not production-ready.


## 2.6 — Chains, routing, and parallel work (Day 14)

Chains connect components. Routing can use a validated category and deterministic Python code. Parallel branches can reduce latency, but still incur the cost of each model call.


In [ ]:
# ▶ Fixed sequence: summarize, then rewrite
summarize = ChatPromptTemplate.from_template("Summarize in one sentence: {text}") | model | StrOutputParser()
rewrite = ChatPromptTemplate.from_template("Rewrite this sentence for a beginner: {summary}") | model | StrOutputParser()
source_text = "An API rate limit caps requests over time to protect a shared service."
summary = summarize.invoke({"text": source_text})
print("Summary:", summary)
print("Rewrite:", rewrite.invoke({"summary": summary}))


In [ ]:
class Route(BaseModel):
    category: Literal["billing", "technical", "other"]
router = model.with_structured_output(Route)
# ▶ The classification is model-generated; routing is enforced by your code.
def route_question(question):
    result = router.invoke("Choose a support category for: " + question)
    instructions = {"billing": "Explain billing concepts", "technical": "Give technical troubleshooting steps", "other": "Answer generally"}
    return result.category, model.invoke(f"{instructions[result.category]} briefly: {question}").text
print(route_question("Why was I charged twice?"))


In [ ]:
from langchain_core.runnables import RunnableParallel
# ▶ Two independent calls followed by a combining call.
branches = RunnableParallel(
    benefits=ChatPromptTemplate.from_template("Give two benefits of {topic}.") | model | StrOutputParser(),
    risks=ChatPromptTemplate.from_template("Give two risks of {topic}.") | model | StrOutputParser(),
)
parts = branches.invoke({"topic": "using AI for customer support"})
print(parts)
print(model.invoke("Combine these into balanced advice: " + json.dumps(parts)).text)


**Try it:** Change the router questions and inspect misclassifications. Compare the sum of branch durations with elapsed time. A fixed chain follows your defined steps; an agent chooses actions dynamically.


## 2.7 — Failures, retries, timeouts, and fallback (Day 15)

Retry transient timeouts, rate limits, and temporary server errors with backoff; do not blindly retry authentication, permission, or malformed-request errors. SDK retries may already be enabled: avoid accidentally multiplying retry attempts. Fallback can increase cost and should be observable. A different model must support the same features and be available to your account.


In [ ]:
from openai import APITimeoutError, APIConnectionError, RateLimitError, InternalServerError
TRANSIENT = (APITimeoutError, APIConnectionError, RateLimitError, InternalServerError)
def invoke_with_fallback(primary, backup, messages):
    try:
        return {"message": primary.invoke(messages), "fallback_used": False, "error_type": None}
    except TRANSIENT as exc:
        print("Primary transient failure:", type(exc).__name__)
        try:
            return {"message": backup.invoke(messages), "fallback_used": True, "error_type": type(exc).__name__}
        except TRANSIENT as backup_exc:
            return {"message": None, "fallback_used": True, "error_type": type(backup_exc).__name__}
# No hidden fallback for authentication, invalid input, or programming errors.
# ▶ Optional: configure another accessible model explicitly before running.
# backup = ChatOpenAI(model=os.environ["OPENAI_FALLBACK_MODEL"], timeout=45, max_retries=1)
# result = invoke_with_fallback(model, backup, "Explain a timeout briefly.")
# print(result)


In [ ]:
# Offline, deterministic fallback test without consuming API credits.
class FailsTransiently:
    def invoke(self, _): raise APIConnectionError(request=None)
class StubBackup:
    def invoke(self, _): return AIMessage(content="Backup response")
test = invoke_with_fallback(FailsTransiently(), StubBackup(), "hello")
assert test["fallback_used"] and test["message"].text == "Backup response"
print("Fallback behavior verified with stubs.")


**Try it:** Simulate both providers failing, and verify your UI displays a friendly message. Separately test a bad key or bad request in a controlled environment: it should surface, not silently trigger retries or fallback. For non-model operations use bounded exponential backoff and idempotency where relevant. Use a rate limiter as well as concurrency limits for provider quotas.


In [ ]:
# Offline deterministic retry exercise for non-model operations.
# In real systems add jitter and an idempotency key for operations with side effects.
def retry_transient(operation, attempts=3, initial_delay=0.1):
    for index in range(attempts):
        try: return operation()
        except (TimeoutError, ConnectionError):
            if index == attempts - 1: raise
            time.sleep(initial_delay * 2 ** index)
attempt_count = [0]
def flaky_demo():
    attempt_count[0] += 1
    if attempt_count[0] < 3: raise TimeoutError("simulated transient error")
    return "success"
print(retry_transient(flaky_demo), "after", attempt_count[0], "attempts")


## 2.8 — Processing at scale (Day 16)

`batch()` makes concurrent ordinary requests; it is different from the provider's asynchronous [Batch API](https://developers.openai.com/api/docs/guides/batch). Limit concurrency and handle individual errors. The resumable example below writes synthetic results as each task finishes, so completed IDs can be skipped after a restart.


In [ ]:
# ▶ Small concurrent example: one API call per input.
texts = ["Great service!", "My order arrived late.", "Where is my invoice?"]
classifier = ChatPromptTemplate.from_template("Classify sentiment as positive, negative, or neutral. Text: {text}") | model | StrOutputParser()
outputs = classifier.batch([{"text": t} for t in texts], config={"max_concurrency": 2}, return_exceptions=True)
for text, output in zip(texts, outputs):
    print(text, "=>", type(output).__name__ if isinstance(output, Exception) else output)


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
# ▶ Optional; writes synthetic results to a local JSONL file. Run again to resume failures only.
def resumable_classify(items, path="module2_batch_results.jsonl", concurrency=2):
    path = Path(path)
    completed = set()
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            row = json.loads(line)
            if row.get("ok"): completed.add(row["id"])
    pending = [item for item in items if item["id"] not in completed]
    def work(item):
        try:
            return {"id": item["id"], "ok": True, "result": classifier.invoke({"text": item["text"]})}
        except Exception as exc:
            return {"id": item["id"], "ok": False, "error_type": type(exc).__name__}
    with ThreadPoolExecutor(max_workers=concurrency) as pool, path.open("a", encoding="utf-8") as f:
        futures = [pool.submit(work, item) for item in pending]
        for future in as_completed(futures):
            row = future.result()
            f.write(json.dumps(row) + "\n"); f.flush()
    print("Previously complete:", len(completed), "Attempted now:", len(pending), "File:", path)
# items = [{"id": str(i), "text": text} for i, text in enumerate(texts)]
# resumable_classify(items)


**Try it:** Estimate total tokens and cost from a representative sample before scaling. Use stable unique IDs, never reuse an ID for changed input, and protect any sensitive output file. Interrupt and rerun the optional job; verify completed IDs are skipped. Failed rows remain in the file as an audit trail. For large jobs add rate limiting, durable storage and handling for partial/corrupt records.


## 2.9 — Cost, logs, and traces (Day 17)

Store metadata rather than raw prompts and outputs. The estimate below uses user-entered rates, excludes cached-input discounts, tools, taxes and other charges, and is **not an invoice**. A missing rate or usage is reported as unknown rather than zero. One trace can contain multiple model and tool calls.


In [ ]:
# Enter current prices per million tokens for YOUR model, or leave as None.
INPUT_USD_PER_MILLION = None
OUTPUT_USD_PER_MILLION = None
def estimate_cost(usage, input_rate=INPUT_USD_PER_MILLION, output_rate=OUTPUT_USD_PER_MILLION):
    if not usage or input_rate is None or output_rate is None: return None
    if usage.get("input_tokens") is None or usage.get("output_tokens") is None: return None
    return (usage["input_tokens"] * input_rate + usage["output_tokens"] * output_rate) / 1_000_000
def logged_invoke(candidate, question, label, path="module2_calls.jsonl"):
    start = time.perf_counter()
    row = {"label": label, "model_configured": MODEL_NAME}
    try:
        response = candidate.invoke(question)
        usage = response.usage_metadata or {}
        row.update(ok=True, usage=usage, cost_usd=estimate_cost(usage), error_type=None)
        return response
    except Exception as exc:
        row.update(ok=False, usage=None, cost_usd=None, error_type=type(exc).__name__)
        raise
    finally:
        row["latency_s"] = round(time.perf_counter() - start, 3)
        with open(path, "a", encoding="utf-8") as f: f.write(json.dumps(row) + "\n")
def report_calls(path="module2_calls.jsonl"):
    rows = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
    if not rows: print("No calls recorded."); return
    known = [r for r in rows if r.get("cost_usd") is not None]
    print("Requests:", len(rows), "Failures:", sum(not r["ok"] for r in rows))
    print("Mean latency:", round(statistics.mean(r["latency_s"] for r in rows), 3), "s")
    print("Slowest:", max(rows, key=lambda r: r["latency_s"]))
    print("Known estimated cost:", sum(r["cost_usd"] for r in known), "USD; unknown cost calls:", len(rows)-len(known))
    if known: print("Most expensive known:", max(known, key=lambda r: r["cost_usd"]))


In [ ]:
# ▶ Optional: ten paid calls, then a report. Use only synthetic inputs.
# for i, question in enumerate(questions, 1):
#     try: logged_invoke(model, question, label=f"q{i}")
#     except Exception as exc: print("Failed:", i, type(exc).__name__)
# report_calls()


In [ ]:
# ▶ Aggregate usage for a small group; metadata support varies.
from langchain_core.callbacks import get_usage_metadata_callback
with get_usage_metadata_callback() as usage_callback:
    model.invoke("Say hello in one sentence.")
    model.invoke("What is 2 + 2?")
print(usage_callback.usage_metadata)


**Try it:** Enter verified prices in the earlier cell, rerun it, then run the ten-call report. Inspect the slowest and most expensive cases. If tracing with [LangSmith](https://docs.langchain.com/langsmith/observability), set its environment variables before initializing the model; traces can include full prompts and outputs. Use synthetic data and review your privacy settings.


## Practice exercises — work before checking

1. A 200-token system message, 50 user tokens and 150 assistant tokens per turn are resent for ten turns. Calculate total input and output tokens. Use current prices of two OpenAI models to estimate costs.
2. For a receipt totaling ₹1,250, classify `1250`, `125`, a nonnumeric string, and `-1250` as correct, valid-but-wrong, or invalid. Apply the schema's own coercion rules when you test.
3. A logged-in user asks to cancel somebody else's account. Where must authorization happen, and what must the tool return on denial?
4. Estimate sequential versus concurrency-10 runtime for 200 calls taking two seconds each. What changes if rate limits cause errors?
5. Modernize an old `LLMChain` / `ConversationBufferMemory` example using `ChatPromptTemplate | model | StrOutputParser` and explicit history.

**Check:** (1) 11,500 input and 1,500 output tokens, ignoring overhead. (2) correct, valid-but-wrong, invalid, invalid. (3) trusted tool/application code checks the authenticated user and denies unauthorized actions. (4) ideal 400 s versus 40 s; reduce concurrency and control request rate when limited. (5) see Unit 2.1 and 2.6. Real tokenization and overhead differ from this arithmetic.


## Practical assessment — small support assistant

This is a teaching prototype using synthetic order data. It classifies first, then streams the final response while handling tool requests. Each model call is captured by the callback for aggregated usage. The assistant uses a bounded loop, an explicit history, validation, timeouts from setup, transient fallback, and per-turn metadata logs. The fallback is disabled until configured. Streaming errors retain partial text and are marked incomplete. Run the setup and tool-loop cells first.


In [ ]:
class Triage(BaseModel):
    category: Literal["order", "billing", "technical", "other"]
    urgency: Literal["low", "medium", "high"]
@tool
def order_status(order_id: str) -> str:
    """Find status for a fictional demo order ID such as ORD-1001."""
    if not re.fullmatch(r"ORD-\d+", order_id): raise ValueError("Invalid order ID")
    orders = {"ORD-1001": "shipped", "ORD-1002": "processing"}
    if order_id not in orders: raise ValueError("Order not found")
    return f"Demo order {order_id}: {orders[order_id]}"
support_tools = [order_status]
support_history = [SystemMessage(content="You are a friendly demo support assistant. Use the order tool for order status; do not invent results. Label demo data clearly.")]
def support_turn(user_text, primary=model, backup=None, path="module2_support_calls.jsonl"):
    start = time.perf_counter()
    record = {"ok": False, "fallback_used": False, "error_type": None, "model_configured": MODEL_NAME}
    # Each attempt restarts this turn from the same committed history.
    def attempt(candidate, callback):
        triage = candidate.with_structured_output(Triage).invoke("Classify this support message: " + user_text)
        print("Triage:", triage.model_dump())
        messages = support_history + [HumanMessage(content=user_text)]
        bound_model = candidate.bind_tools(support_tools)
        # Resolve any requested tools before streaming a final answer.
        for _ in range(4):
            response = bound_model.invoke(messages)
            messages.append(response)
            if not response.tool_calls:
                # The final response is streamed from the established conversation, without re-running tools.
                messages.pop()
                chunks = []
                try:
                    for chunk in candidate.stream(messages + [SystemMessage(content="Answer the user's latest question now using any tool results above. Do not call tools.")]):
                        chunks.append(chunk.text)
                        print(chunk.text, end="", flush=True)
                except (Exception, KeyboardInterrupt) as exc:
                    return triage, "".join(chunks), False, type(exc).__name__
                return triage, "".join(chunks), True, None
            for call in response.tool_calls:
                try:
                    if call["name"] != order_status.name: raise ValueError("Unknown tool")
                    messages.append(order_status.invoke(call))
                except Exception as exc:
                    messages.append(ToolMessage(content=f"Tool error: {type(exc).__name__}; request could not be completed.", tool_call_id=call["id"], name=call["name"]))
        return triage, "Sorry, I could not complete that request.", False, "ToolRoundLimit"
    usage_data = {}
    try:
        with get_usage_metadata_callback() as cb:
            try:
                triage, text, complete, error = attempt(primary, cb)
            except TRANSIENT as exc:
                record["error_type"] = type(exc).__name__
                if backup is None: raise
                record["fallback_used"] = True
                triage, text, complete, error = attempt(backup, cb)
        usage_data = cb.usage_metadata
        record.update(ok=complete, error_type=error, triage=triage.model_dump())
        if complete:
            support_history.extend([HumanMessage(content=user_text), AIMessage(content=text)])
        else:
            print("\n[Incomplete response; not added to history]")
        return text
    except TRANSIENT as exc:
        record["error_type"] = type(exc).__name__
        print("Sorry, the service is temporarily unavailable. Please try again.")
        return None
    finally:
        record["latency_s"] = round(time.perf_counter() - start, 3)
        record["usage_by_model"] = usage_data
        # No raw messages or user identifiers are written.
        with open(path, "a", encoding="utf-8") as f: f.write(json.dumps(record) + "\n")


In [ ]:
# ▶ Run manually, then test with ten synthetic questions. Each turn makes multiple calls.
# support_turn("What is the status of demo order ORD-1001?")
# support_turn("What did I just ask about?")
# support_turn("Check ORD-9999.")


In [ ]:
# Report support turns and their aggregated usage by model. Set current per-model prices first.
SUPPORT_RATES = {}  # e.g. {"actual-reported-model-id": (input_usd_per_million, output_usd_per_million)}
def report_support(path="module2_support_calls.jsonl", rates=None):
    rates = rates or {}
    rows = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
    if not rows: print("No turns recorded."); return
    total, unknown = 0.0, 0
    for row in rows:
        usage = row.get("usage_by_model") or {}
        if not usage: unknown += 1
        for model_id, counts in usage.items():
            if model_id not in rates or counts.get("input_tokens") is None or counts.get("output_tokens") is None:
                unknown += 1; continue
            input_rate, output_rate = rates[model_id]
            total += (counts["input_tokens"]*input_rate + counts["output_tokens"]*output_rate)/1_000_000
    print("Turns:", len(rows), "Incomplete:", sum(not r["ok"] for r in rows),
          "Fallbacks:", sum(r["fallback_used"] for r in rows))
    print("Mean seconds:", round(statistics.mean(r["latency_s"] for r in rows), 3),
          "Slowest:", max(r["latency_s"] for r in rows))
    print("Known estimated USD:", total, "Unknown usage/model entries:", unknown)
# report_support(rates=SUPPORT_RATES)


**Complete the assessment:** (1) Test ten different questions, including an unknown order and a follow-up. (2) Configure a compatible fallback model and inject failures to verify both fallback and friendly failure behavior. (3) Add per-model current rates and compute a cost report from `usage_by_model` in `module2_support_calls.jsonl`; report unknown rates separately. (4) Write a paragraph on failures and fixes. Triage and preliminary tool calls are included in aggregate usage when reported; interrupted requests may lack usage, so the report is an estimate. The final streamed generation is an additional call and can differ from the preliminary no-tool response. The example is educational, not an authenticated order system.

### Self-check

- Why must conversation history be resent, and how does it affect cost?
- Why can valid structured output still be incorrect?
- Who runs tools, and where do authorization and limits belong?
- Which failures are retryable? What is the difference between concurrent `batch()` and a provider Batch API?
- What does a trace reveal that individual request logs do not?

### References

- [OpenAI model catalog](https://developers.openai.com/api/docs/models), [pricing](https://developers.openai.com/api/docs/pricing), [Python SDK](https://developers.openai.com/api/reference/python)
- [LangChain ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai), [models](https://docs.langchain.com/oss/python/langchain/models), [tools](https://docs.langchain.com/oss/python/langchain/tools), [streaming](https://docs.langchain.com/oss/python/langchain/streaming)
- [Pydantic validation](https://docs.pydantic.dev/latest/concepts/validators/)
